In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import AdaBoostRegressor

# 1. Define your exact feature and target setup
verde_core_features = [
    'Temp_Verde', 'Precip_Verde', 'Temp_Maricopa', 'Precip_Maricopa',
    'Population', 'IrrigatedLand_Acres', 'datacenters_TotalMW', 'datacenters_TotalNum'
]

ada_lags = [
    'log_flow_verde_downstream_lag_1', 'log_flow_verde_downstream_lag_12',
    'log_flow_verde_upstream_lag_8', 'log_flow_verde_upstream_lag_4', 
    'log_flow_verde_upstream_lag_10'
]

features = verde_core_features + ada_lags
target = 'log_flow_verde_downstream'

# 2. Clean the data using your unique lags checklist
all_cols_needed = features + [target]
df_model_train = df_train.dropna(subset=all_cols_needed).reset_index(drop=True)

# 3. Define Bootstrap Parameters
# A block size of 30 preserves 1-month chronological relationships (lags 1-12)
block_size = 30  
n_bootstraps = 100  
n_samples = len(df_model_train)
n_blocks_needed = int(np.ceil(n_samples / block_size))

# Array to hold predictions from all 100 bootstrap models
# Rows = test samples, Columns = bootstrap iterations
all_bootstrap_preds = np.zeros((n_samples, n_bootstraps))

print(f"Bootstrapping {n_bootstraps} variations of AdaBoost...")

for i in range(n_bootstraps):
    # Fix random seed per iteration for reproducibility, but vary it across loops
    np.random.seed(123 + i)
    
    # Draw random block starting positions
    start_indices = np.random.randint(0, n_samples - block_size, size=n_blocks_needed)
    
    # Stitch blocks together to build the bootstrapped training set
    bootstrap_indices = []
    for start in start_indices:
        bootstrap_indices.extend(list(range(start, start + block_size)))
    
    # Clip to original length to match exact dataset size
    bootstrap_indices = bootstrap_indices[:n_samples]
    df_bootstrapped = df_model_train.iloc[bootstrap_indices]
    
    X_train_b = df_bootstrapped[features]
    y_train_b = df_bootstrapped[target]
    
    # Initialize your best performing model hyperparameters
    model_b = AdaBoostRegressor(learning_rate=0.01, n_estimators=100, random_state=123 + i)
    model_b.fit(X_train_b, y_train_b)
    
    # Predict on the original continuous dataset to see prediction variations
    all_bootstrap_preds[:, i] = model_b.predict(df_model_train[features])

# 4. Compute upper and lower bounds (95% Confidence / Prediction Interval)
# 2.5th percentile is the lower bound, 97.5th percentile is the upper bound
lower_bounds = np.percentile(all_bootstrap_preds, 2.5, axis=1)
upper_bounds = np.percentile(all_bootstrap_preds, 97.5, axis=1)
mean_predictions = np.mean(all_bootstrap_preds, axis=1)

# 5. Package into a clean DataFrame for inspection or plotting
df_results = pd.DataFrame({
    'Actual_Flow': df_model_train[target],
    'Predicted_Flow_Mean': mean_predictions,
    'Lower_Bound_95': lower_bounds,
    'Upper_Bound_95': upper_bounds
})

print("\nBootstrap complete! Sample of results with confidence bounds:")
print(df_results.head(10))